# RAG Pipeline — Ancient Egypt Document Assistant

## 2.1 Load & Inspect

This section loads and inspects the source documents used for the RAG system.

The dataset consists of HTML documents containing information about Ancient Egypt.
Before building the RAG pipeline, the documents are checked to verify that they can be read successfully and that their content is suitable for text extraction.

In [1]:
from pathlib import Path
from bs4 import BeautifulSoup
import re
import chromadb
from sentence_transformers import SentenceTransformer

In [2]:
html_path = Path("../Data/ancient_egypt.html")

with open(html_path, "r", encoding="utf-8") as f:
    html = f.read()

print("HTML loaded successfully")
print("Characters:", len(html))

HTML loaded successfully
Characters: 700857


In [3]:
soup = BeautifulSoup(html, "html.parser")

# Remove unnecessary elements
for element in soup(["script", "style", "noscript"]):
    element.decompose()

text = soup.get_text(separator=" ")

# Clean whitespace
text = re.sub(r"\s+", " ", text).strip()

print("Text extracted successfully")
print("Characters after cleaning:", len(text))
print("\nPreview:")
print(text[:1000])

Text extracted successfully
Characters after cleaning: 599023

Preview:
The Project Gutenberg eBook of Ancient Egypt, by George Rawlinson, M.A.. The Project Gutenberg eBook of Ancient Egypt This eBook is for the use of anyone anywhere in the United States and most other parts of the world at no cost and with almost no restrictions whatsoever. You may copy it, give it away or re-use it under the terms of the Project Gutenberg License included with this eBook or online at www.gutenberg.org . If you are not located in the United States, you will have to check the laws of the country where you are located before using this eBook. Title : Ancient Egypt Author : George Rawlinson Arthur Gilman Release date : April 20, 2005 [eBook #15663] Most recently updated: December 14, 2020 Language : English Other information and formats : www.gutenberg.org/ebooks/15663 Credits : Produced by Juliet Sutherland, Susan Skinner and Distributed Proofreaders Europe at http://dp.rastko.net. *** START OF THE PRO

In [4]:
#Basic inspection

words = text.split()

print("Number of words:", len(words))
print("Number of characters:", len(text))
print("First 50 words:")
print(words[:50])

Number of words: 101880
Number of characters: 599023
First 50 words:
['The', 'Project', 'Gutenberg', 'eBook', 'of', 'Ancient', 'Egypt,', 'by', 'George', 'Rawlinson,', 'M.A..', 'The', 'Project', 'Gutenberg', 'eBook', 'of', 'Ancient', 'Egypt', 'This', 'eBook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'in', 'the', 'United', 'States', 'and', 'most', 'other', 'parts', 'of', 'the', 'world', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever.', 'You', 'may', 'copy']


In [5]:
soup = BeautifulSoup(html, "html.parser")

# Show all headings
headings = soup.find_all(["h1", "h2", "h3"])

print("Number of headings:", len(headings))

for i, heading in enumerate(headings[:30]):
    print(i, "->", heading.get_text(" ", strip=True))

Number of headings: 93
0 -> The Project Gutenberg eBook of Ancient Egypt
1 -> ANCIENT EGYPT
2 -> BY
3 -> GEORGE RAWLINSON, M.A.
4 -> ARTHUR GILMAN, M.A. AUTHOR OF "THE STORY OF ROME," ETC.
5 -> CONTENTS.
6 -> I.
7 -> II.
8 -> III.
9 -> IV.
10 -> V.
11 -> VI.
12 -> VII.
13 -> VIII.
14 -> IX.
15 -> X.
16 -> XI.
17 -> XII.
18 -> XIII.
19 -> XIV.
20 -> XV.
21 -> XVI.
22 -> XVII.
23 -> XVIII.
24 -> XIX.
25 -> XX.
26 -> XXI.
27 -> XXII.
28 -> XXIII.
29 -> XXIV.


### Dataset Inspection Summary

- Number of source documents: 1
- Format: HTML
- Number of pages: Not applicable, since the source is an HTML document rather than a paginated PDF.
- Text extraction: Successful
- OCR required: No
- Cleaning performed: Removed HTML formatting, scripts, styles, and extra whitespace.
- Number of words: 101880
- Number of characters: 599023

In [6]:
# Display all headings with their text
for i, heading in enumerate(headings):
    title = heading.get_text(" ", strip=True)
    print(f"{i}: {title}")

0: The Project Gutenberg eBook of Ancient Egypt
1: ANCIENT EGYPT
2: BY
3: GEORGE RAWLINSON, M.A.
4: ARTHUR GILMAN, M.A. AUTHOR OF "THE STORY OF ROME," ETC.
5: CONTENTS.
6: I.
7: II.
8: III.
9: IV.
10: V.
11: VI.
12: VII.
13: VIII.
14: IX.
15: X.
16: XI.
17: XII.
18: XIII.
19: XIV.
20: XV.
21: XVI.
22: XVII.
23: XVIII.
24: XIX.
25: XX.
26: XXI.
27: XXII.
28: XXIII.
29: XXIV.
30: XXV.
31: XXVI.
32: XXVII.
33: INDEX 403
34: LIST OF ILLUSTRATIONS.
35: THE STORY OF ANCIENT EGYPT.
36: I.
37: THE LAND OF EGYPT.
38: II.
39: THE PEOPLE OF EGYPT.
40: III.
41: THE DAWN OF HISTORY.
42: IV.
43: THE PYRAMID BUILDERS.
44: V.
45: THE RISE OF THEBES TO POWER, AND THE EARLY THEBAN KINGS.
46: VI.
47: THE GOOD AMENEMHAT AND HIS WORKS.
48: VII.
49: ABRAHAM IN EGYPT.
50: VIII.
51: THE GREAT INVASION—THE HYKSÔS OR SHEPHERD KINGS—JOSEPH AND APEPI.
52: IX.
53: HOW THE HYKSÔS WERE EXPELLED FROM EGYPT.
54: X.
55: THOTHMES I., THE FIRST GREAT EGYPTIAN CONQUEROR.
56: XI.
57: QUEEN HATASU AND HER MERCHANT FLEET.
58

In [7]:
# Inspect the headings around the actual chapters

for i, heading in enumerate(headings):
    title = heading.get_text(" ", strip=True)

    if title in [
        "I.",
        "II.",
        "III.",
        "IV.",
        "V."
    ]:
        print(i, repr(title), heading.name)

6 'I.' h3
7 'II.' h3
8 'III.' h3
9 'IV.' h3
10 'V.' h3
36 'I.' h2
38 'II.' h2
40 'III.' h2
42 'IV.' h2
44 'V.' h2


In [8]:
for i, heading in enumerate(headings):
    title = heading.get_text(" ", strip=True)

    if len(title) <= 5:
        print(i, repr(title), heading.name)

2 'BY' h3
6 'I.' h3
7 'II.' h3
8 'III.' h3
9 'IV.' h3
10 'V.' h3
11 'VI.' h3
12 'VII.' h3
13 'VIII.' h3
14 'IX.' h3
15 'X.' h3
16 'XI.' h3
17 'XII.' h3
18 'XIII.' h3
19 'XIV.' h3
20 'XV.' h3
21 'XVI.' h3
22 'XVII.' h3
24 'XIX.' h3
25 'XX.' h3
26 'XXI.' h3
27 'XXII.' h3
29 'XXIV.' h3
30 'XXV.' h3
31 'XXVI.' h3
36 'I.' h2
38 'II.' h2
40 'III.' h2
42 'IV.' h2
44 'V.' h2
46 'VI.' h2
48 'VII.' h2
50 'VIII.' h2
52 'IX.' h2
54 'X.' h2
56 'XI.' h2
58 'XII.' h2
60 'XIII.' h2
62 'XIV.' h2
64 'XV.' h2
66 'XVI.' h2
68 'XVII.' h2
72 'XIX.' h2
74 'XX.' h2
76 'XXI.' h2
78 'XXII.' h2
82 'XXIV.' h2
84 'XXV.' h2
86 'XXVI.' h2


In [9]:
for heading in soup.find_all("h2"):
    title = heading.get_text(" ", strip=True)

    if title in [
        "I.", "II.", "III.", "IV.", "V.", "VI.", "VII.",
        "VIII.", "IX.", "X.", "XI.", "XII.", "XIII.", "XIV.",
        "XV.", "XVI.", "XVII.", "XVIII.", "XIX.", "XX.", "XXI.",
        "XXII.", "XXIII.", "XXIV.", "XXV.", "XXVI.", "XXVII."
    ]:
        print("=" * 60)
        print("CHAPTER:", title)
        
        # Print the next few HTML elements
        for element in list(heading.find_all_next())[:8]:
            print(element.name, ":", element.get_text(" ", strip=True)[:150])

CHAPTER: I.
a : 
h3 : THE LAND OF EGYPT.
p : In shape Egypt is like a lily with a crooked stem. A broad blossom
terminates it at its upper end; a button of a bud projects from the
stalk a little 
a : 
p : At the first glance, the country seems to divide itself into two
strongly contrasted regions; and this was the original impression which
it made upon 
a : 
p : Such is the twofold division of the country which impresses the observer
strongly at the first. On a longer sojourn and a more intimate
familiarity, t
a : 
CHAPTER: II.
a : 
h3 : THE PEOPLE OF EGYPT.
p : Where the Egyptians came from, is a difficult question to answer.
Ancient speculators, when they could not derive a people definitely from
any other, 
p : It is generally answered that they came from Asia; but this is not much
more than a conjecture. The physical type of the Egyptians is different
from t
a : 
p : Still, whencesoever derived, the Egyptian people, as it existed in the
flourishing times of Egyptian history, was be

In [10]:
from langchain_core.documents import Document

# Find all actual chapter headings
chapter_headings = []

for h2 in soup.find_all("h2"):
    chapter_number = h2.get_text(" ", strip=True)

    if chapter_number in [
        "I.", "II.", "III.", "IV.", "V.", "VI.", "VII.",
        "VIII.", "IX.", "X.", "XI.", "XII.", "XIII.", "XIV.",
        "XV.", "XVI.", "XVII.", "XVIII.", "XIX.", "XX.",
        "XXI.", "XXII.", "XXIII.", "XXIV.", "XXV.", "XXVI.", "XXVII."
    ]:
        chapter_headings.append(h2)

print("Number of chapters found:", len(chapter_headings))

Number of chapters found: 27


In [11]:
chapters = []

for i, h2 in enumerate(chapter_headings):

    chapter_number = h2.get_text(" ", strip=True)

    # Chapter title
    h3 = h2.find_next("h3")
    chapter_title = h3.get_text(" ", strip=True) if h3 else "Unknown Title"

    # Collect elements until the next chapter h2
    elements = []

    for element in h2.find_all_next():

        # Stop when we reach the next chapter
        if element.name == "h2" and element in chapter_headings:
            break

        if element.name == "p":
            text = element.get_text(" ", strip=True)

            if text:
                elements.append(text)

    chapter_text = "\n\n".join(elements)

    chapters.append({
        "chapter": chapter_number,
        "title": chapter_title,
        "text": chapter_text
    })

print("Extracted chapters:", len(chapters))

Extracted chapters: 27


In [12]:
for chapter in chapters[:3]:
    print("=" * 70)
    print("Chapter:", chapter["chapter"])
    print("Title:", chapter["title"])
    print("Characters:", len(chapter["text"]))
    print("Preview:")
    print(chapter["text"][:500])

Chapter: I.
Title: THE LAND OF EGYPT.
Characters: 29224
Preview:
In shape Egypt is like a lily with a crooked stem. A broad blossom
terminates it at its upper end; a button of a bud projects from the
stalk a little below the blossom, on the left-hand side. The broad
blossom is the Delta, extending from Aboosir to Tineh, a direct distance
of a hundred and eighty miles, which the projection of the coast—the
graceful swell of the petals—enlarges to two hundred and thirty. The
bud is the Fayoum, a natural depression in the hills that shut in the
Nile valley on th
Chapter: II.
Title: THE PEOPLE OF EGYPT.
Characters: 32273
Preview:
Where the Egyptians came from, is a difficult question to answer.
Ancient speculators, when they could not derive a people definitely from
any other, took refuge in the statement, or the figment, that they were
the children of the soil which they had always occupied. Modern
theorists may say, if it please them, that they were evolved out of the
monkeys that had th

### Chapter-Based Document Extraction

The HTML document was parsed and divided into its 27 main chapters based on the chapter headings.
For each chapter, we extracted its title and text content while preserving the chapter structure. This structure will be used in the chunking stage to keep related information together and provide meaningful source metadata for retrieval.


### Chunking Strategy

The extracted chapters were split into smaller chunks using `RecursiveCharacterTextSplitter` with a chunk size of 1000 characters and an overlap of 150 characters. Paragraphs and sentences were prioritized as split points to preserve the context of the text. Each chunk also retains metadata such as the chapter number, chapter title, source file, and chunk ID for better retrieval and source tracking.


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

print("Chunk size:", text_splitter._chunk_size)
print("Chunk overlap:", text_splitter._chunk_overlap)

Chunk size: 1000
Chunk overlap: 150


In [14]:
all_chunks = []

for chapter in chapters:

    chapter_chunks = text_splitter.split_text(chapter["text"])

    for chunk_id, chunk in enumerate(chapter_chunks):

        all_chunks.append({
            "text": chunk,
            "chapter": chapter["chapter"],
            "title": chapter["title"],
            "chunk_id": chunk_id
        })

print("Total chunks:", len(all_chunks))

Total chunks: 753


In [15]:
for chunk in all_chunks[:3]:
    print("=" * 70)
    print("Chapter:", chunk["chapter"])
    print("Title:", chunk["title"])
    print("Chunk ID:", chunk["chunk_id"])
    print("Characters:", len(chunk["text"]))
    print(chunk["text"][:500])

Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 0
Characters: 965
In shape Egypt is like a lily with a crooked stem. A broad blossom
terminates it at its upper end; a button of a bud projects from the
stalk a little below the blossom, on the left-hand side. The broad
blossom is the Delta, extending from Aboosir to Tineh, a direct distance
of a hundred and eighty miles, which the projection of the coast—the
graceful swell of the petals—enlarges to two hundred and thirty. The
bud is the Fayoum, a natural depression in the hills that shut in the
Nile valley on th
Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 1
Characters: 220
Delta, sometimes not more than a mile broad, never more than eight or
ten miles. No other country in the world is so strangely shaped, so
long compared to its width, so straggling, so hard to govern from a
single centre.
Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 2
Characters: 960
At the first glance, the country seems to divide itself into two
strongly contra

In [16]:
documents = []

for chunk in all_chunks:

    doc = Document(
        page_content=chunk["text"],
        metadata={
            "source": "ancient_egypt.html",
            "chapter": chunk["chapter"],
            "title": chunk["title"],
            "chunk_id": chunk["chunk_id"]
        }
    )

    documents.append(doc)

print("Number of Documents:", len(documents))

Number of Documents: 753


In [17]:
print(documents[0])

page_content='In shape Egypt is like a lily with a crooked stem. A broad blossom
terminates it at its upper end; a button of a bud projects from the
stalk a little below the blossom, on the left-hand side. The broad
blossom is the Delta, extending from Aboosir to Tineh, a direct distance
of a hundred and eighty miles, which the projection of the coast—the
graceful swell of the petals—enlarges to two hundred and thirty. The
bud is the Fayoum, a natural depression in the hills that shut in the
Nile valley on the west, which has been rendered cultivable for many
thousands of years by the introduction into it of the Nile water,
through a canal known as the "Bahr Yousouf." The long stalk of the lily
is the Nile valley itself, which is a ravine scooped in the rocky soil
for seven hundred miles from the First Cataract to the apex of the
Delta, sometimes not more than a mile broad, never more than eight or
ten miles. No other country in the world is so strangely shaped, so' metadata={'source':

In [18]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


In [19]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 8s [Retry 5/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD h

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 4s [Retry 3/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 8s [Retry 4/5].
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json
Retrying in 8s [Retry 5/5].


Embedding model loaded successfully.


In [20]:
persist_directory = Path("../Data/chroma_db")

vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=str(persist_directory)
)

print("Vector store created successfully.")
print("Number of documents:", len(documents))
print("Persisted at:", persist_directory)

Vector store created successfully.
Number of documents: 753
Persisted at: ..\Data\chroma_db


In [21]:
query = "What is the Nile valley?"

results = vectordb.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print("=" * 70)
    print("Result:", i)
    print("Chapter:", doc.metadata["chapter"])
    print("Title:", doc.metadata["title"])
    print("Chunk ID:", doc.metadata["chunk_id"])
    print("Text:", doc.page_content[:300])

Result: 1
Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 8
Text: the maps, nothing more than the valley and plain watered by the Nile,
for nearly seven hundred miles by the river's course from the
Mediterranean southwards." [1] The great wastes on either side of the
Nile valley are in no sense Egypt, neither the un dulating sandy desert
to the west, nor the rocky
Result: 2
Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 8
Text: the maps, nothing more than the valley and plain watered by the Nile,
for nearly seven hundred miles by the river's course from the
Mediterranean southwards." [1] The great wastes on either side of the
Nile valley are in no sense Egypt, neither the un dulating sandy desert
to the west, nor the rocky
Result: 3
Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 15
Text: Though thus useful, beneficent, and indeed essential to the existence of
Egypt, the Nile can scarcely be said to add much to the variety of the
landscape or to the beauty of the scenery. It is someth

### Retrieval and Prompting

The persisted Chroma vector store is used to retrieve the most relevant document chunks for a given question. The retrieved context is then provided to the language model to generate grounded answers. Each answer will include the relevant chapter as a source to improve traceability and reduce unsupported information.


In [22]:
questions = [
    "What is the geographical shape of Egypt?",
    "What are the main regions of Egypt?",
    "Where did the ancient Egyptians come from?",
    "What was the role of the Nile in Egypt?",
    "Who were the Hyksos?",
    "How were the Hyksos expelled from Egypt?",
    "Who was Queen Hatasu?",
    "What was the religion of Khuenaten?",
    "Who was Psamatik I?",
    "How did the Persian conquest of Egypt occur?"
]

print("Number of questions:", len(questions))

Number of questions: 10


In [23]:
for question in questions:
    results = vectordb.similarity_search(question, k=2)

    print("=" * 80)
    print("QUESTION:", question)

    for i, doc in enumerate(results, 1):
        print(f"\nResult {i}")
        print("Chapter:", doc.metadata["chapter"])
        print("Title:", doc.metadata["title"])
        print("Chunk ID:", doc.metadata["chunk_id"])
        print("Text:", doc.page_content[:400])

QUESTION: What is the geographical shape of Egypt?

Result 1
Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 7
Text: It may be objected to this description, that the Egypt which it presents
to the reader is not the Egypt of the maps. Undoubtedly it is not. The
maps give the name of Egypt to a broad rectangular space which they mark
out in the north-eastern corner of Africa, bounded on two sides by the
Mediterranean and the Red Sea, and on the two others by two imaginary
lines which the map-makers kindly draw for

Result 2
Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 7
Text: It may be objected to this description, that the Egypt which it presents
to the reader is not the Egypt of the maps. Undoubtedly it is not. The
maps give the name of Egypt to a broad rectangular space which they mark
out in the north-eastern corner of Africa, bounded on two sides by the
Mediterranean and the Red Sea, and on the two others by two imaginary
lines which the map-makers kindly draw for
QUESTION: What 

In [24]:
def retrieve_documents(question, k=3):
    results = vectordb.similarity_search(question, k=k)
    return results

In [25]:
question = "What was the role of the Nile in Egypt?"

results = retrieve_documents(question, k=3)

for i, doc in enumerate(results, 1):
    print("=" * 70)
    print(f"Result {i}")
    print("Chapter:", doc.metadata["chapter"])
    print("Title:", doc.metadata["title"])
    print("Chunk ID:", doc.metadata["chunk_id"])
    print("Text:", doc.page_content[:500])

Result 1
Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 10
Text: was evident from the projection of the shore of the Delta beyond the
general coast-line of Africa eastward and westward; and, he added, "I am
convinced, for my own part, that if the Nile should please to divert his
waters from their present bed into the Red Sea, he would fill it up and
turn it into dry land in the space of twenty thousand years, or maybe in
half that time—for he is a mighty river and a most energetic one."
Here, in this last expression, he is thoroughly right, though the method

Result 2
Chapter: I.
Title: THE LAND OF EGYPT.
Chunk ID: 10
Text: was evident from the projection of the shore of the Delta beyond the
general coast-line of Africa eastward and westward; and, he added, "I am
convinced, for my own part, that if the Nile should please to divert his
waters from their present bed into the Red Sea, he would fill it up and
turn it into dry land in the space of twenty thousand years, or maybe in
half th

In [26]:
def build_context(documents):
    context_parts = []

    for doc in documents:
        context_parts.append(
            f"[Source: Chapter {doc.metadata['chapter']} - "
            f"{doc.metadata['title']} | "
            f"Chunk {doc.metadata['chunk_id']}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(context_parts)

In [27]:
question = "What was the role of the Nile in Egypt?"

retrieved_docs = retrieve_documents(question, k=3)

context = build_context(retrieved_docs)

print(context[:3000])

[Source: Chapter I. - THE LAND OF EGYPT. | Chunk 10]
was evident from the projection of the shore of the Delta beyond the
general coast-line of Africa eastward and westward; and, he added, "I am
convinced, for my own part, that if the Nile should please to divert his
waters from their present bed into the Red Sea, he would fill it up and
turn it into dry land in the space of twenty thousand years, or maybe in
half that time—for he is a mighty river and a most energetic one."
Here, in this last expression, he is thoroughly right, though the method
of the Nile's energy has been other than he supposed. The Nile, working
from its immense reservoirs in the equatorial regions, has gradually
scooped itself out a deep bed in the sand and rock of the desert, which
must have originally extended across the whole of northern Africa from
the Atlantic to the Red Sea. Having scooped itself out this bed to a
depth, in places, of three hundred feet from the desert level, it has
then proceeded partially